# bridgechem — a hard-sphere gas in a box

Part 1: an ideal-ish gas of hard spheres that fly ballistically and collide
elastically with each other and the walls. `run()` computes the whole trajectory
up front (fast, numba-accelerated), then displays it with **play / pause / scrub**
controls (an `ipywidgets.Play` widget) — no separate `show()` step, no HTML file,
and you can pause and drag the slider back to inspect a specific collision.

Everything is computed in **SI units**. Lengths are entered in **nm** and masses
in **amu** for convenience; results (speeds, temperature, pressure) come back in SI.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import bridgechem as bc

# Fewer, larger particles are easiest to watch. Particles are auto-sized to fill
# ~10% of the box, and drawn at their true collision size.
system = bc.box(N=200, size=(40, 40), temperature=300)
system

## Run and watch it

`run` integrates the system and hands back an interactive player. Pass
`vectors=True` to draw each particle's velocity arrow. It still returns a
`Simulation` object holding the trajectory for analysis afterwards.

Real gas speeds (hundreds of m/s) would fly past in a blur, so playback is
paced by a `speed` knob rather than shown at true speed: at the default
`speed=1` a typical particle takes a few seconds to cross the box -- slow
enough to actually follow collisions. Try `speed=0.3` (slower) or `speed=3`
(faster); it only changes the *display* pace, not the underlying physics.

In [ ]:
sim = system.run(steps=20000, vectors=True, speed=1.0)

## A light/heavy mixture, coloured by mass

`set_mass` lets you build a mixture -- here, a quarter of the particles become
four times heavier. Colouring by mass (instead of speed) makes it easy to track
which particles are which as they mix.

In [ ]:
mixture = bc.box(N=200, size=(40, 40), temperature=300, gas="helium")
mixture.set_mass(gas="argon", indices=slice(0, 50))  # first 50 particles become argon
sim_mix = mixture.run(steps=15000, color_by="mass")

## Speed distribution vs Maxwell–Boltzmann, and why N matters

The histogram of speeds should match the 2D Maxwell–Boltzmann (Rayleigh)
distribution. A *single* measurement of a small system is noisy -- exactly like a
real experiment with too few molecules. Run the same setup at a few values of `N`
and compare a **single snapshot** (`frame=-1`, not pooled over time) against the
theoretical curve: the larger `N` is, the less that one measurement fluctuates
around Maxwell–Boltzmann. This is the actual "law of large numbers" content behind
statistical mechanics -- more informative than watching one system relax once.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True, sharey=True)
for ax, N in zip(axes, [30, 150, 800]):
    s = bc.box(N=N, size=(60, 60), temperature=300, seed=0)
    sim_n = s.run(steps=15000, animate=False)
    sim_n.histogram("speeds", frame=-1, ax=ax)   # one instant, not pooled -- shows real sampling noise
    ax.set_title(f"N = {N}")
plt.tight_layout()
plt.show()

(`sim.histogram(..., frame="all")` pools every recorded frame together for a
smoother-looking plot -- that's a *different*, also valid idea: for a system in
equilibrium, time-averaging one simulation approximates the same distribution as
averaging over many independent systems, the ergodic hypothesis.)

## Pressure: three ways to compute it

`bridgechem` can compute pressure three different ways, each teaching something
different about where "pressure" actually comes from:

- **`method="wall"`** -- momentum transferred to the container walls per unit time
  and length. The direct, operational definition: literally what a pressure gauge
  bolted to the wall would read. Needs reflective walls.
- **`method="virial"`** -- the Clausius virial theorem applied to particle-particle
  collisions/forces: $P = \big[N k_B T + \tfrac{1}{2}\langle\sum r_{ij}\cdot F_{ij}\rangle\big]/A$.
  Works with *or without* walls (the standard technique for periodic simulations),
  and should agree with the wall method for a reflective box -- two independent
  routes to the same physical quantity.
- **`method="ideal"`** -- the textbook estimate $P = Nk_BT/A$: a theoretical
  reference, not something measured from the dynamics at all (no particle size, no
  collisions).

In [ ]:
system = bc.box(N=250, size=(40, 40), temperature=300, packing=0.10, seed=0)
sim = system.run(steps=20000, animate=False)

p_wall = sim.calculate("pressure", method="wall")
p_virial = sim.calculate("pressure", method="virial")
p_ideal = sim.calculate("pressure", method="ideal")
print(f"wall   = {p_wall:.4e} N/m")
print(f"virial = {p_virial:.4e} N/m   (ratio to wall: {p_virial / p_wall:.3f} -- should be close to 1)")
print(f"ideal  = {p_ideal:.4e} N/m   (Z = P/P_ideal = {p_wall / p_ideal:.3f} -- excluded-area effect)")

Only `method="virial"` works for a periodic box -- there's no wall to measure
momentum transfer at, so `method="wall"` raises a clear error instead of silently
returning nonsense. The bulk pressure it reports should still match the reflective
case above, since a homogeneous gas's pressure doesn't care how the boundary is
implemented.

In [ ]:
periodic = bc.box(N=250, size=(40, 40), temperature=300, packing=0.10,
                   boundary="periodic", seed=0)
sim_p = periodic.run(steps=20000, animate=False)
print("periodic virial pressure:", sim_p.calculate("pressure"))  # method="virial" by default here

try:
    sim_p.calculate("pressure", method="wall")
except ValueError as e:
    print("\nmethod='wall' on a periodic box raises:\n ", e)

You can also replay a recorded simulation at any time with `sim.show()` (also with
play/pause controls, no HTML), overriding the display size or coloring for that call:

In [ ]:
sim.show(display_scale=1.5, color_by="speed")

# Part 2 — interactions and phase transitions

So far particles have been hard spheres with no forces between them (an ideal
gas). `add_interactions("LJ")` switches on Lennard-Jones forces and moves the
engine to velocity-Verlet integration -- the steep repulsive core of LJ keeps
particles apart continuously, so there's no separate collision step anymore.
Periodic boundaries are recommended for interacting systems (the standard
choice for bulk gas/liquid MD).

In [ ]:
interacting = bc.box(N=200, size=(6.8, 6.8), gas="argon", temperature=300,
                     boundary="periodic", seed=0)
interacting.add_interactions("LJ")   # epsilon/sigma default to argon's tabulated LJ parameters
sim_lj = interacting.run(steps=20000, color_by="speed")

## Energy conservation: the key sanity check

With real forces at play, kinetic energy alone isn't conserved -- but kinetic +
potential should be, to high precision, since velocity-Verlet is symplectic.
This is the standard way to catch a broken force law or too-large a time step.

In [ ]:
E = sim_lj.calculate("total_energy")
print(f"total energy drift: {(E.max() - E.min()) / abs(E.mean()):.2e}  (should be well under 1%)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sim_lj.times * 1e12, sim_lj.calculate("kinetic_energy"), label="kinetic")
ax.plot(sim_lj.times * 1e12, sim_lj.calculate("potential_energy"), label="potential")
ax.plot(sim_lj.times * 1e12, E, label="total", lw=2, color="k")
ax.set_xlabel("t (ps)"); ax.set_ylabel("energy (J)"); ax.legend()
plt.show()

## Cooling a gas: watching a phase transition

`set_temperature(target, rate=...)` ramps the temperature during the *next*
`run()` -- rate is in K/ps, and omitting it jumps to the target immediately
instead. An *ideal* gas has no phase transition (nothing for particles to bind
to), so this is really a demonstration of what interactions add: cool the same
LJ argon box from 300 K down to 20 K and watch it condense. The potential
energy per particle becoming much more negative is the condensation signature
-- particles are finding and settling into each other's attractive wells.

In [ ]:
cooling = bc.box(N=200, size=(6.8, 6.8), gas="argon", temperature=300,
                 boundary="periodic", seed=0)
cooling.add_interactions("LJ")
cooling.set_temperature(target_temperature=20, rate=30)   # 300 K -> 20 K at 30 K/ps
sim_cool = cooling.run(steps=40000, color_by="speed")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(sim_cool.times * 1e12, sim_cool.calculate("temperature"))
axes[0].set_xlabel("t (ps)"); axes[0].set_ylabel("temperature (K)")
axes[0].set_title("cooling ramp")

axes[1].plot(sim_cool.times * 1e12, sim_cool.calculate("potential_energy") / cooling.N)
axes[1].set_xlabel("t (ps)"); axes[1].set_ylabel("potential energy / particle (J)")
axes[1].set_title("condensation: particles bind together as it cools")
plt.tight_layout()
plt.show()

Compare the *start* (hot, gas-like: particles spread out and moving fast) and
the *end* (cold, liquid/solid-like: particles clustered and slow) of that run
by scrubbing `sim_cool.show()`'s slider back to frame 0, or replaying:

In [ ]:
sim_cool.show(color_by="speed")

## Coming next

- Custom pairwise potentials beyond Lennard-Jones.
- 3D boxes.